# Task 1

In [1]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

grid = [
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 1],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0]
]

rows = len(grid)
cols = len(grid[0])

start = (1, 1)
target = (4, 4)

K = 4  # steps

# Variables
row = [model.NewIntVar(0, rows - 1, f"r{t}") for t in range(K)]
col = [model.NewIntVar(0, cols - 1, f"c{t}") for t in range(K)]

# Start
model.Add(row[0] == start[0])
model.Add(col[0] == start[1])

# End
model.Add(row[K-1] == target[0])
model.Add(col[K-1] == target[1])

for t in range(K - 1):
    b1 = model.NewBoolVar(f"m1_{t}")  # (-1,-1)
    b2 = model.NewBoolVar(f"m2_{t}")  # (-1,1)
    b3 = model.NewBoolVar(f"m3_{t}")  # (1,-1)
    b4 = model.NewBoolVar(f"m4_{t}")  # (1,1)

    # exactly one move
    model.Add(b1 + b2 + b3 + b4 == 1)

    # apply moves
    model.Add(row[t+1] == row[t] - 1).OnlyEnforceIf(b1)
    model.Add(col[t+1] == col[t] - 1).OnlyEnforceIf(b1)

    model.Add(row[t+1] == row[t] - 1).OnlyEnforceIf(b2)
    model.Add(col[t+1] == col[t] + 1).OnlyEnforceIf(b2)

    model.Add(row[t+1] == row[t] + 1).OnlyEnforceIf(b3)
    model.Add(col[t+1] == col[t] - 1).OnlyEnforceIf(b3)

    model.Add(row[t+1] == row[t] + 1).OnlyEnforceIf(b4)
    model.Add(col[t+1] == col[t] + 1).OnlyEnforceIf(b4)

# Avoid obstacles
for t in range(K):
    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1:
                model.AddForbiddenAssignments([row[t], col[t]], [(i, j)])

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
    for t in range(K):
        print(f"Step {t}: ({solver.Value(row[t])}, {solver.Value(col[t])})")
else:
    print("No solution found")

No solution found


# Task 2

In [2]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

grid = [
    [0,1,1,0],
    [1,1,0,0],
    [0,1,1,0],
    [0,0,0,0]
]

rows = len(grid)
cols = len(grid[0])

# Variables
x = [[model.NewIntVar(0,1,f"x[{i}][{j}]") for j in range(cols)] for i in range(rows)]

# Fix values
for i in range(rows):
    for j in range(cols):
        model.Add(x[i][j] == grid[i][j])

# Perimeter variables
edges = []

for i in range(rows):
    for j in range(cols):
        if grid[i][j] == 1:

            # check 4 directions
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni = i + di
                nj = j + dj

                e = model.NewIntVar(0,1,f"edge_{i}_{j}_{di}_{dj}")

                # If outside grid → edge = 1
                if ni < 0 or ni >= rows or nj < 0 or nj >= cols:
                    model.Add(e == 1)

                else:
                    # edge = 1 if neighbor is water
                    model.Add(e == 1).OnlyEnforceIf(x[ni][nj].Not())
                    model.Add(e == 0).OnlyEnforceIf(x[ni][nj])

                edges.append(e)

# Total perimeter
perimeter = model.NewIntVar(0, 1000, "perimeter")
model.Add(perimeter == sum(edges))

solver = cp_model.CpSolver()
status = solver.Solve(model)

# Output
if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
    print("Perimeter:", solver.Value(perimeter))
else:
    print("No solution")

Perimeter: 14


# Task 3

In [3]:
from ortools.sat.python import cp_model

def solve_tsp(distance):
    n = len(distance)
    model = cp_model.CpModel()

    arc = [[model.new_bool_var(f"arc_{i}_{j}") for j in range(n)] for i in range(n)]

    u = [model.new_int_var(0, n-1, f"u_{i}") for i in range(n)]

    for i in range(n):
        model.add(sum(arc[i][j] for j in range(n)) == 1)

    for j in range(n):
        model.add(sum(arc[i][j] for i in range(n)) == 1)

    for i in range(n):
        model.add(arc[i][i] == 0)

    model.add(u[0] == 0)
    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                model.add(u[j] >= u[i] + 1 - n * (1 - arc[i][j]))

    total_distance = sum(
        arc[i][j] * distance[i][j]
        for i in range(n)
        for j in range(n)
    )
    model.minimize(total_distance)

    solver = cp_model.CpSolver()
    status = solver.solve(model)

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(f"Total distance: {solver.objective_value}")
        route = [0]
        current = 0
        for _ in range(n - 1):
            for j in range(n):
                if solver.value(arc[current][j]) == 1:
                    route.append(j)
                    current = j
                    break
        route.append(0)
        print(f"Route: {' → '.join(map(str, route))}")
    else:
        print("No solution found")

distance = [
    [0, 12, 10, 19, 8, 14, 17, 21, 11, 13],
    [12, 0, 3, 7, 6, 20, 9, 18, 14, 15],
    [10, 3, 0, 6, 5, 11, 13, 16, 9, 12],
    [19, 7, 6, 0, 4, 10, 8, 15, 17, 14],
    [8, 6, 5, 4, 0, 9, 7, 12, 10, 11],
    [14, 20, 11, 10, 9, 0, 6, 13, 15, 16],
    [17, 9, 13, 8, 7, 6, 0, 5, 12, 10],
    [21, 18, 16, 15, 12, 13, 5, 0, 14, 9],
    [11, 14, 9, 17, 10, 15, 12, 14, 0, 8],
    [13, 15, 12, 14, 11, 16, 10, 9, 8, 0]
]
solve_tsp(distance)

Total distance: 72.0
Route: 0 → 8 → 9 → 7 → 6 → 5 → 3 → 2 → 1 → 4 → 0
